In [5]:
!pip install pyspark==3.5.1 \
    boto3 \
    findspark


Defaulting to user installation because normal site-packages is not writeable
  Using cached pyspark-3.5.1.tar.gz (317.0 MB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 2.5 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=7a8ec3dd4e3650a9f60ab3e396ded1afb65df9effb0745542d42bc799ce66773
  Stored in directory: /home/datadisk/jupyter-admin/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:

import findspark, os
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("local-test")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(spark.version)
spark.range(5).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 12:05:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.1
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [1]:
# check psql
import psycopg2

con = ICEBERG_JDBC_URL#'jdbc:postgresql://89.208.223.38:5432/demo_db'

conn = psycopg2.connect(
    host='89.208.208.210',
    port=5432,
    dbname='demo_db',
    user=ICEBERG_JDBC_USER,
    password=ICEBERG_JDBC_PASSWORD,
)

cursor = conn.cursor()
cursor.execute("SELECT version();")

print(cursor.fetchone())

cursor.close()
conn.close()

NameError: name 'ICEBERG_JDBC_URL' is not defined

In [10]:
q = '''
CREATE TABLE IF NOT EXISTS iceberg_tables (
    catalog_name VARCHAR(255) NOT NULL,
    table_namespace VARCHAR(255) NOT NULL,
    table_name VARCHAR(255) NOT NULL,
    metadata_location VARCHAR(1000),
    previous_metadata_location VARCHAR(1000),
    PRIMARY KEY (catalog_name, table_namespace, table_name)
);

CREATE TABLE IF NOT EXISTS iceberg_namespace_properties (
    catalog_name VARCHAR(255) NOT NULL,
    namespace VARCHAR(255) NOT NULL,
    property_key VARCHAR(255) NOT NULL,
    property_value VARCHAR(1000),
    PRIMARY KEY (catalog_name, namespace, property_key)
);
'''

with psycopg2.connect(
    host='89.208.208.210',
    port=5432,
    dbname='demo_db',
    user=ICEBERG_JDBC_USER,
    password=ICEBERG_JDBC_PASSWORD,
) as conn:
    with conn.cursor() as curs:
        curs.execute(q)

NameError: name 'psycopg2' is not defined

In [2]:
S3_ENDPOINT = "https://hb.ru-msk.vkcloud-storage.ru"      # пример: https://hb.bizmrg.com
S3_ACCESS_KEY = 'dCAMn72g45jebuu8GmrpJZ'
S3_SECRET_KEY = '3X9vrJkEx9LXBXTYwTycPe6z2RYekoTDKpRnGWXG537i'
S3_BUCKET = "iceberg-mcs809"
ICEBERG_JDBC_URL = 'jdbc:postgresql://89.208.208.210:5432/demo_db'
ICEBERG_JDBC_USER = "user"
ICEBERG_JDBC_PASSWORD = "9Kc*345C91L7zhoD8"
ICEBERG_CATALOG_NAME = "ice"

In [15]:
spark.stop()

In [34]:
!pwd

/home/datadisk/jupyter-admin/iceberg


In [3]:
import os

JARS_DIR = "/home/datadisk/jupyter-admin/iceberg"

ICEBERG_JAR = f"{JARS_DIR}/iceberg-spark-runtime-3.5_2.12-1.7.1.jar"
POSTGRES_JAR = f"{JARS_DIR}/postgresql-42.7.4.jar"
HADOOP_AWS_JAR = f"{JARS_DIR}/hadoop-aws-3.3.4.jar"
AWS_SDK_JAR = f"{JARS_DIR}/aws-java-sdk-bundle-1.12.751.jar"

jars = [
    ICEBERG_JAR,
    POSTGRES_JAR,
    HADOOP_AWS_JAR,
    AWS_SDK_JAR,
]

jars_comma = ",".join(jars)
jars_colon = ":".join(jars)

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--jars {jars_comma} "
    f"--driver-class-path {jars_colon} "
    f"pyspark-shell"
)

In [4]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

In [5]:
import findspark
import os
#findspark.init()

from pyspark.sql import SparkSession

# ---- ваши параметры ----

WAREHOUSE_PATH = f"s3a://{S3_BUCKET}/iceberg"



# Maven coordinates для Iceberg и JDBC-драйвера
ICEBERG_VERSION = "1.7.1"
JDBC_DRIVER = "org.postgresql:postgresql:42.7.4"

# Для Spark 3.5 используется модуль iceberg-spark-runtime-3.5
# ICEBERG_PACKAGES = [
#     f"iceberg-spark-runtime-3.5_2.13-1.7.1.jar",
#     f"org.apache.iceberg:iceberg-aws:{ICEBERG_VERSION}",  # опционально, если будете использовать aws-интеграцию

#     "com.amazonaws:aws-java-sdk-bundle:1.12.751",    
#     JDBC_DRIVER,
# ]

JARS_DIR = "/home/datadisk/jupyter-admin/iceberg"

ICEBERG_JAR = os.path.join(JARS_DIR, f"iceberg-spark-runtime-3.5_2.12-1.7.1.jar")
POSTGRES_JAR = os.path.join(JARS_DIR, "postgresql-42.7.4.jar")
HADOOP_AWS_JAR = os.path.join(JARS_DIR, "hadoop-aws-3.3.4.jar")
AWS_SDK_JAR = os.path.join(JARS_DIR, "aws-java-sdk-bundle-1.12.751.jar")

ALL_JARS = ",".join([POSTGRES_JAR, HADOOP_AWS_JAR, AWS_SDK_JAR, ICEBERG_JAR])

ALL_JARS_LIST = [POSTGRES_JAR, HADOOP_AWS_JAR, AWS_SDK_JAR, ICEBERG_JAR]
ALL_JARS_CP = ":".join(ALL_JARS_LIST) 

print("Using jars:", ALL_JARS)

spark = (
    SparkSession.builder
    .appName("LocalSpark-Iceberg-VKCloud")
    .master("local[2]")
    
    # Джарников оч много разных. Надежнее скачать и положить локально рабочую сборку
    .config("spark.jars", ALL_JARS)
    .config("spark.driver.extraClassPath", ALL_JARS_CP)         # classpath драйвера
    .config("spark.executor.extraClassPath", ALL_JARS_CP)  
    
    #.config("spark.jars.packages", ",".join(ICEBERG_PACKAGES))

    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.uri", ICEBERG_JDBC_URL)
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.jdbc.user", ICEBERG_JDBC_USER)
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.jdbc.password", ICEBERG_JDBC_PASSWORD)
    .config(f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.warehouse", WAREHOUSE_PATH)

    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")

    # S3 и S3A - РАЗНЫЕ вещи. Смотря что прописано в location в метаданных айсберга - нужны разные либы чтобы это читать
    # Проще сразу прописать в конф все варианты
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

    .config("spark.sql.shuffle.partitions", "2")

    # Для того чтобы работали CALL и другие процедуры поддержки
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate()
)

spark


Using jars: /home/datadisk/jupyter-admin/iceberg/postgresql-42.7.4.jar,/home/datadisk/jupyter-admin/iceberg/hadoop-aws-3.3.4.jar,/home/datadisk/jupyter-admin/iceberg/aws-java-sdk-bundle-1.12.751.jar,/home/datadisk/jupyter-admin/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.1.jar


26/05/21 12:06:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

try:
    spark.sparkContext._jvm.java.lang.Class.forName("org.apache.iceberg.spark.SparkCatalog")
    print("OK: SparkCatalog class is visible")
except Exception as e:
    print("FAIL: SparkCatalog class is NOT visible")
    raise

Spark: 3.5.1
Scala: 2.12.18
OK: SparkCatalog class is visible


In [6]:
cl = spark.sparkContext._jvm.Thread.currentThread().getContextClassLoader()
cl.loadClass("org.apache.iceberg.spark.SparkCatalog")
print("OK via context classloader")

OK via context classloader


In [27]:
spark.stop()

In [26]:
!ls -lh /home/datadisk/jupyter-admin/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.1.jar
!jar tf /home/datadisk/jupyter-admin/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.1.jar | grep 'org/apache/iceberg/spark/SparkCatalog.class'

-rw-r--r-- 1 jupyter-admin jupyter-admin 41M Feb 26 11:58 /home/datadisk/jupyter-admin/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.1.jar
org/apache/iceberg/spark/SparkCatalog.class


In [8]:
from pyspark.sql.functions import col, upper

data = [("alice", 5), ("bob", 3), ("carol", 7)]
df = spark.createDataFrame(data, ["name", "experience"])

df_tr = (
    df
    .withColumn("name_upper", upper(col("name")))
    .filter(col("experience") >= 4)
)

df_tr.show()


+-----+----------+----------+
| name|experience|name_upper|
+-----+----------+----------+
|alice|         5|     ALICE|
|carol|         7|     CAROL|
+-----+----------+----------+



## Спарк не всегда регистрирует в себе каталоги
Вызвав листинг каталогов можно не увидеть подключенный айсберга. 
При этом остальной функционал будет работать (!)

In [7]:
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/home/datadisk/jupyter-admin/iceberg/spark-warehouse')]

In [6]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [6]:
# Проверяем что все нужные конфиги доехали вызвались

for k, v in sorted(spark.sparkContext.getConf().getAll()):
    if "spark.sql.catalog" in k:
        print(k, "=", v)


spark.sql.catalog.ice = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.ice.catalog-impl = org.apache.iceberg.jdbc.JdbcCatalog
spark.sql.catalog.ice.jdbc.password = 9Kc*345C91L7zhoD8
spark.sql.catalog.ice.jdbc.user = user
spark.sql.catalog.ice.uri = jdbc:postgresql://89.208.208.210:5432/demo_db
spark.sql.catalog.ice.warehouse = s3a://iceberg-mcs809/iceberg


In [24]:
print("Spark version:", spark.version)

print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

Spark version: 3.5.1
Spark: 3.5.1
Scala: 2.12.18


In [7]:
spark.sql("SHOW NAMESPACES IN ice").show(truncate=False)

+---------+
|namespace|
+---------+
|analytics|
+---------+



26/05/21 12:06:59 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


## Создаем таблицы

In [8]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS ice.analytics")

DataFrame[]

In [9]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ice.analytics.events (
    event_id          STRING NOT NULL COMMENT 'Уникальный идентификатор события',
    event_ts          TIMESTAMP NOT NULL COMMENT 'Время события в UTC',
    event_date        DATE COMMENT 'Дата события, можно использовать для удобной фильтрации',

    event_type        STRING NOT NULL COMMENT 'Тип события: page_view, click, purchase, login и т.д.',
    event_name        STRING COMMENT 'Бизнес-название события',
    source            STRING COMMENT 'Источник события: web, ios, android, backend, api',

    user_id           STRING COMMENT 'Идентификатор пользователя',
    anonymous_id      STRING COMMENT 'Анонимный идентификатор до логина',
    session_id        STRING COMMENT 'Идентификатор сессии',

    page_url          STRING COMMENT 'URL страницы',
    page_title        STRING COMMENT 'Заголовок страницы',
    referrer_url      STRING COMMENT 'URL источника перехода',

    utm_source        STRING COMMENT 'UTM source',
    utm_medium        STRING COMMENT 'UTM medium',
    utm_campaign      STRING COMMENT 'UTM campaign',
    utm_content       STRING COMMENT 'UTM content',
    utm_term          STRING COMMENT 'UTM term',

    ip_address        STRING COMMENT 'IP адрес',
    user_agent        STRING COMMENT 'User-Agent',
    device_type       STRING COMMENT 'desktop, mobile, tablet, bot',
    os_name           STRING COMMENT 'Операционная система',
    browser_name      STRING COMMENT 'Браузер',

    country           STRING COMMENT 'Страна',
    region            STRING COMMENT 'Регион',
    city              STRING COMMENT 'Город',

    product_id        STRING COMMENT 'Идентификатор товара/объекта',
    category_id       STRING COMMENT 'Категория товара/объекта',
    order_id          STRING COMMENT 'Идентификатор заказа, если событие связано с заказом',

    revenue           DECIMAL(18, 4) COMMENT 'Выручка по событию',
    currency          STRING COMMENT 'Валюта',

    properties        MAP<STRING, STRING> COMMENT 'Дополнительные свойства события',
    payload           STRING COMMENT 'Сырой JSON payload, если нужен',

    ingestion_ts      TIMESTAMP NOT NULL COMMENT 'Время загрузки события в хранилище',
    producer          STRING COMMENT 'Сервис или пайплайн, отправивший событие',
    schema_version    INT COMMENT 'Версия схемы события'
)
USING iceberg
PARTITIONED BY (
    days(event_ts)
)
TBLPROPERTIES (
    'format-version' = '2',

    'write.format.default' = 'parquet',
    'write.parquet.compression-codec' = 'zstd',

    'write.target-file-size-bytes' = '134217728',

    'write.distribution-mode' = 'hash',

    'commit.retry.num-retries' = '5',
    'commit.retry.min-wait-ms' = '1000',
    'commit.retry.max-wait-ms' = '60000',

    'history.expire.max-snapshot-age-ms' = '604800000',
    'history.expire.min-snapshots-to-keep' = '3',

    'write.metadata.delete-after-commit.enabled' = 'true',
    'write.metadata.previous-versions-max' = '10'
)
""")

26/05/21 12:07:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/21 12:07:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [10]:
spark.sql("""
INSERT INTO ice.analytics.events VALUES (
    'evt_000001',
    TIMESTAMP '2026-05-21 12:00:00',
    DATE '2026-05-21',

    'page_view',
    'Product Page Viewed',
    'web',

    'user_123',
    'anon_abc',
    'sess_789',

    'https://example.com/products/42',
    'Product 42',
    'https://google.com',

    'google',
    'cpc',
    'spring_campaign',
    'banner_a',
    'running shoes',

    '127.0.0.1',
    'Mozilla/5.0',
    'desktop',
    'macOS',
    'Chrome',

    'RU',
    'Moscow',
    'Moscow',

    'product_42',
    'category_7',
    NULL,

    CAST(NULL AS DECIMAL(18, 4)),
    NULL,

    map('color', 'black', 'size', '42'),
    '{"event":"page_view","product_id":"product_42"}',

    current_timestamp(),
    'test_producer',
    1
)
""")

DataFrame[]

In [11]:
spark.sql("""
SELECT
    event_id,
    event_ts,
    event_type,
    event_name,
    source,
    user_id,
    session_id,
    page_url,
    properties
FROM ice.analytics.events
""").show(truncate=False)

+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+
|event_id  |event_ts           |event_type|event_name         |source|user_id |session_id|page_url                       |properties                  |
+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+
|evt_000001|2026-05-21 12:00:00|page_view |Product Page Viewed|web   |user_123|sess_789  |https://example.com/products/42|{color -> black, size -> 42}|
+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+



In [12]:
spark.sql("SELECT * FROM ice.analytics.events.snapshots").show(truncate=False)
spark.sql("SELECT * FROM ice.analytics.events.history").show(truncate=False)
spark.sql("SELECT * FROM ice.analytics.events.files").show(truncate=False)
spark.sql("SELECT * FROM ice.analytics.events.partitions").show(truncate=False)
spark.sql("CALL ice.system.rewrite_data_files(table => 'analytics.events')").show(truncate=False)
spark.sql("CALL ice.system.rewrite_manifests(table => 'analytics.events')").show(truncate=False)

+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+
|event_id  |event_ts           |event_type|event_name         |source|user_id |session_id|page_url                       |properties                  |
+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+
|evt_000001|2026-05-21 12:00:00|page_view |Product Page Viewed|web   |user_123|sess_789  |https://example.com/products/42|{color -> black, size -> 42}|
+----------+-------------------+----------+-------------------+------+--------+----------+-------------------------------+----------------------------+



In [13]:
spark.sql("""
SELECT *
FROM ice.analytics.events.history
ORDER BY made_current_at DESC
""").show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-05-21 12:07:39.039|1245056847513738605|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+



In [14]:
spark.sql("""
SELECT *
FROM ice.analytics.events.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                              |summary                                                                                                                                                          

In [15]:
# Размер и файлы

spark.sql("""
SELECT
    COUNT(*) AS data_files,
    SUM(record_count) AS total_records,
    ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) AS total_size_mb,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) AS avg_file_size_mb,
    MIN(file_size_in_bytes) AS min_file_size_bytes,
    MAX(file_size_in_bytes) AS max_file_size_bytes
FROM ice.analytics.events.files
WHERE content = 0
""").show(truncate=False)

+----------+-------------+-------------+----------------+-------------------+-------------------+
|data_files|total_records|total_size_mb|avg_file_size_mb|min_file_size_bytes|max_file_size_bytes|
+----------+-------------+-------------+----------------+-------------------+-------------------+
|1         |1            |0.01         |0.01            |11990              |11990              |
+----------+-------------+-------------+----------------+-------------------+-------------------+



In [16]:
# Детектор мелких файлов

spark.sql("""
SELECT
    file_path,
    record_count,
    ROUND(file_size_in_bytes / 1024 / 1024, 2) AS file_size_mb,
    partition
FROM ice.analytics.events.files
WHERE content = 0
  AND file_size_in_bytes < 32 * 1024 * 1024
ORDER BY file_size_in_bytes ASC
""").show(100, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------+
|file_path                                                                                                                              |record_count|file_size_mb|partition   |
+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------+
|s3a://iceberg-mcs809/iceberg/analytics/events/data/event_ts_day=2026-05-21/00000-1-391e436b-192f-4259-9b10-02ecff5f4943-0-00001.parquet|1           |0.01        |{2026-05-21}|
+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------+



In [ ]:
# Тайм Тревел

df = (
    spark.read
    .option("snapshot-id", "PUT_SNAPSHOT_ID_HERE")
    .table("ice.analytics.events")
)

df.show(truncate=False)

In [17]:
spark.sql("""
SELECT *
FROM ice.analytics.events TIMESTAMP AS OF '2026-05-21 12:00:00'
""").show(truncate=False)

IllegalArgumentException: Cannot find a snapshot older than 2026-05-21T12:00:00+00:00

In [18]:
spark.sql("""
CALL ice.system.rewrite_data_files(
    table => 'analytics.events',
    where => 'event_ts >= TIMESTAMP "2026-05-01 00:00:00"',
    options => map(
        'target-file-size-bytes', '134217728'
    )
)
""").show(truncate=False)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'CALL'.(line 2, pos 0)

== SQL ==

CALL ice.system.rewrite_data_files(
^^^
    table => 'analytics.events',
    where => 'event_ts >= TIMESTAMP "2026-05-01 00:00:00"',
    options => map(
        'target-file-size-bytes', '134217728'
    )
)


# КЕЙС С DBT

In [16]:
spark.sql("SHOW TABLES IN ice.dbt_test").show(truncate=False)

Py4JJavaError: An error occurred while calling o237.sql.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'ice': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:53)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:53)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndNamespace$.unapply(LookupCatalog.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:51)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:30)
	at org.a

In [8]:
spark.sql("SELECT * FROM ice.dbt_test.customer").show(5)

25/12/24 09:03:40 INFO V2ScanRelationPushDown: 
Output: c_customer_sk#132L, c_customer_id#133, c_current_cdemo_sk#134L, c_current_hdemo_sk#135L, c_current_addr_sk#136L, c_first_shipto_date_sk#137L, c_first_sales_date_sk#138L, c_salutation#139, c_first_name#140, c_last_name#141, c_preferred_cust_flag#142, c_birth_day#143, c_birth_month#144, c_birth_year#145, c_birth_country#146, c_login#147, c_email_address#148, c_last_review_date_sk#149L
         
25/12/24 09:03:40 INFO SnapshotScan: Scanning table ice.dbt_test.customer snapshot 5624943814538454095 created at 2025-12-23T19:00:24.040+00:00 with filter true
25/12/24 09:03:40 INFO BaseDistributedDataScan: Planning file tasks locally for table ice.dbt_test.customer
25/12/24 09:03:40 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 4 partition(s) for table ice.dbt_test.customer
25/12/24 09:03:40 INFO MemoryStore: Block broadcast_3 stored as values in memory (estimated size 32.0 KiB, free 434.2 MiB)
25/12/24 09:03:40 INFO 

+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+--------------------+--------------------+---------------------+-----------+-------------+------------+---------------+-------+--------------------+---------------------+
|c_customer_sk|   c_customer_id|c_current_cdemo_sk|c_current_hdemo_sk|c_current_addr_sk|c_first_shipto_date_sk|c_first_sales_date_sk|c_salutation|        c_first_name|         c_last_name|c_preferred_cust_flag|c_birth_day|c_birth_month|c_birth_year|c_birth_country|c_login|     c_email_address|c_last_review_date_sk|
+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+--------------------+--------------------+---------------------+-----------+-------------+------------+---------------+-------+--------------------+---------------------+
|     11673989|AAAAAAAAFIBCCLAA|             9065

25/12/24 09:03:41 INFO CodecPool: Got brand-new decompressor [.zstd]
25/12/24 09:03:41 INFO Executor: Finished task 0.0 in stage 1.0 (TID 1). 5584 bytes result sent to driver
25/12/24 09:03:41 INFO TaskSetManager: Finished task 0.0 in stage 1.0 (TID 1) in 881 ms on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local (executor driver) (1/1)
25/12/24 09:03:41 INFO TaskSchedulerImpl: Removed TaskSet 1.0, whose tasks have all completed, from pool 
25/12/24 09:03:41 INFO DAGScheduler: ResultStage 1 (showString at NativeMethodAccessorImpl.java:0) finished in 0.888 s
25/12/24 09:03:41 INFO DAGScheduler: Job 1 is finished. Cancelling potential speculative or zombie tasks for this job
25/12/24 09:03:41 INFO TaskSchedulerImpl: Killing all running tasks in stage 1: Stage finished
25/12/24 09:03:41 INFO DAGScheduler: Job 1 finished: showString at NativeMethodAccessorImpl.java:0, took 0.895489 s


In [5]:
print(spark.sparkContext._jsc.hadoopConfiguration().get("fs.s3a.impl"))
print(spark.sparkContext._jsc.hadoopConfiguration().get("fs.s3.impl"))


org.apache.hadoop.fs.s3a.S3AFileSystem
org.apache.hadoop.fs.s3a.S3AFileSystem


In [16]:
df = spark.table("ice.tpcds.item")

25/12/24 09:17:32 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3://iceberg-mcs809/iceberg/tpcds/item-fbe4d683402e43a29068b4549e86f1e7/metadata/00002-e86b54ad-d11e-4a20-8389-98aa14cbc7d4.metadata.json
25/12/24 09:17:32 INFO BaseMetastoreCatalog: Table loaded by catalog: ice.tpcds.item


In [17]:
df.printSchema()

root
 |-- i_item_sk: long (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: date (nullable = true)
 |-- i_rec_end_date: date (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: decimal(7,2) (nullable = true)
 |-- i_wholesale_cost: decimal(7,2) (nullable = true)
 |-- i_brand_id: integer (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: integer (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: integer (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: integer (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: integer (nullable = true)
 |-- i_product_name: string (nullable = true)



In [15]:
df.show(5)

25/12/24 09:15:09 INFO V2ScanRelationPushDown: 
Output: c_customer_sk#264L, c_customer_id#265, c_current_cdemo_sk#266L, c_current_hdemo_sk#267L, c_current_addr_sk#268L, c_first_shipto_date_sk#269L, c_first_sales_date_sk#270L, c_salutation#271, c_first_name#272, c_last_name#273, c_preferred_cust_flag#274, c_birth_day#275, c_birth_month#276, c_birth_year#277, c_birth_country#278, c_login#279, c_email_address#280, c_last_review_date_sk#281L
         
25/12/24 09:15:09 INFO SnapshotScan: Scanning table ice.dbt_test.customer snapshot 5624943814538454095 created at 2025-12-23T19:00:24.040+00:00 with filter true
25/12/24 09:15:09 INFO BaseDistributedDataScan: Planning file tasks locally for table ice.dbt_test.customer
25/12/24 09:15:09 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 4 partition(s) for table ice.dbt_test.customer
25/12/24 09:15:09 INFO MemoryStore: Block broadcast_6 stored as values in memory (estimated size 32.0 KiB, free 434.2 MiB)
25/12/24 09:15:09 INFO 

+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+--------------------+--------------------+---------------------+-----------+-------------+------------+---------------+-------+--------------------+---------------------+
|c_customer_sk|   c_customer_id|c_current_cdemo_sk|c_current_hdemo_sk|c_current_addr_sk|c_first_shipto_date_sk|c_first_sales_date_sk|c_salutation|        c_first_name|         c_last_name|c_preferred_cust_flag|c_birth_day|c_birth_month|c_birth_year|c_birth_country|c_login|     c_email_address|c_last_review_date_sk|
+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+--------------------+--------------------+---------------------+-----------+-------------+------------+---------------+-------+--------------------+---------------------+
|     11673989|AAAAAAAAFIBCCLAA|             9065

25/12/24 09:15:11 INFO CodecPool: Got brand-new decompressor [.zstd]
25/12/24 09:15:11 INFO Executor: Finished task 0.0 in stage 2.0 (TID 2). 5584 bytes result sent to driver
25/12/24 09:15:11 INFO TaskSetManager: Finished task 0.0 in stage 2.0 (TID 2) in 1265 ms on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local (executor driver) (1/1)
25/12/24 09:15:11 INFO TaskSchedulerImpl: Removed TaskSet 2.0, whose tasks have all completed, from pool 
25/12/24 09:15:11 INFO DAGScheduler: ResultStage 2 (showString at NativeMethodAccessorImpl.java:0) finished in 1.271 s
25/12/24 09:15:11 INFO DAGScheduler: Job 2 is finished. Cancelling potential speculative or zombie tasks for this job
25/12/24 09:15:11 INFO TaskSchedulerImpl: Killing all running tasks in stage 2: Stage finished
25/12/24 09:15:11 INFO DAGScheduler: Job 2 finished: showString at NativeMethodAccessorImpl.java:0, took 1.276252 s


In [19]:
from pyspark.sql import functions as F

# пример: добавить возраст клиента по дате создания (условный пример)
res = (
    df
    .withColumn("item_desc_upper", F.upper("i_item_desc"))
    .withColumn("last_year", F.year("i_rec_start_date"))
    .filter(F.col("i_category_id") == 3)
)

res.show(5)


25/12/24 09:19:17 INFO V2ScanRelationPushDown: 
Pushing operators to ice.tpcds.item
Pushed Filters: i_category_id IS NOT NULL, i_category_id = 3
Post-Scan Filters: isnotnull(i_category_id#429),(i_category_id#429 = 3)
         
25/12/24 09:19:17 INFO V2ScanRelationPushDown: 
Output: i_item_sk#418L, i_item_id#419, i_rec_start_date#420, i_rec_end_date#421, i_item_desc#422, i_current_price#423, i_wholesale_cost#424, i_brand_id#425, i_brand#426, i_class_id#427, i_class#428, i_category_id#429, i_category#430, i_manufact_id#431, i_manufact#432, i_size#433, i_formulation#434, i_color#435, i_units#436, i_container#437, i_manager_id#438, i_product_name#439
         
25/12/24 09:19:17 INFO SnapshotScan: Scanning table ice.tpcds.item snapshot 5313358808039063702 created at 2025-11-24T16:00:43.700+00:00 with filter (i_category_id IS NOT NULL AND i_category_id = (1-digit-int))
25/12/24 09:19:17 INFO BaseDistributedDataScan: Planning file tasks locally for table ice.tpcds.item
25/12/24 09:19:18 INFO 

+---------+----------------+----------------+--------------+--------------------+---------------+----------------+----------+--------------------+----------+--------------------+-------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+----------+-----------+------------+--------------------+--------------------+---------+
|i_item_sk|       i_item_id|i_rec_start_date|i_rec_end_date|         i_item_desc|i_current_price|i_wholesale_cost|i_brand_id|             i_brand|i_class_id|             i_class|i_category_id|          i_category|i_manufact_id|          i_manufact|              i_size|       i_formulation|             i_color|   i_units|i_container|i_manager_id|      i_product_name|     item_desc_upper|last_year|
+---------+----------------+----------------+--------------+--------------------+---------------+----------------+----------+--------------------+----------+--------------------+-------------+--------

25/12/24 09:19:19 INFO Executor: Finished task 0.0 in stage 3.0 (TID 3). 7282 bytes result sent to driver
25/12/24 09:19:19 INFO TaskSetManager: Finished task 0.0 in stage 3.0 (TID 3) in 985 ms on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local (executor driver) (1/1)
25/12/24 09:19:19 INFO TaskSchedulerImpl: Removed TaskSet 3.0, whose tasks have all completed, from pool 
25/12/24 09:19:19 INFO DAGScheduler: ResultStage 3 (showString at NativeMethodAccessorImpl.java:0) finished in 1.026 s
25/12/24 09:19:19 INFO DAGScheduler: Job 3 is finished. Cancelling potential speculative or zombie tasks for this job
25/12/24 09:19:19 INFO TaskSchedulerImpl: Killing all running tasks in stage 3: Stage finished
25/12/24 09:19:19 INFO DAGScheduler: Job 3 finished: showString at NativeMethodAccessorImpl.java:0, took 1.030252 s
25/12/24 09:19:19 INFO CodeGenerator: Code generated in 16.097318 ms            


In [20]:
agg = (
    res
    .groupBy("i_manufact")
    .agg(
        F.countDistinct("i_item_sk").alias("items_cnt"),
        F.count("*").alias("rows_cnt")
    )
)

agg.show(20, truncate=False)

25/12/24 09:21:16 INFO V2ScanRelationPushDown: 
Pushing operators to ice.tpcds.item
Pushed Filters: i_category_id IS NOT NULL, i_category_id = 3
Post-Scan Filters: isnotnull(i_category_id#429),(i_category_id#429 = 3)
         
25/12/24 09:21:16 INFO V2ScanRelationPushDown: 
Output: i_item_sk#418L, i_category_id#429, i_manufact#432
         
25/12/24 09:21:16 INFO SnapshotScan: Scanning table ice.tpcds.item snapshot 5313358808039063702 created at 2025-11-24T16:00:43.700+00:00 with filter (i_category_id IS NOT NULL AND i_category_id = (1-digit-int))
25/12/24 09:21:16 INFO BaseDistributedDataScan: Planning file tasks locally for table ice.tpcds.item
25/12/24 09:21:16 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table ice.tpcds.item
25/12/24 09:21:16 INFO MemoryStore: Block broadcast_12 stored as values in memory (estimated size 32.0 KiB, free 434.2 MiB)
25/12/24 09:21:16 INFO MemoryStore: Block broadcast_12_piece0 stored as bytes in memory (estima

+--------------------------------------------------+---------+--------+
|i_manufact                                        |items_cnt|rows_cnt|
+--------------------------------------------------+---------+--------+
|n stationese                                      |21       |21      |
|barprically                                       |32       |32      |
|ationeing                                         |58       |58      |
|callycally                                        |48       |48      |
|callyoughtanti                                    |25       |25      |
|n stought                                         |46       |46      |
|eingantiought                                     |63       |63      |
|priought                                          |67       |67      |
|n stableable                                      |41       |41      |
|callyantiable                                     |48       |48      |
|antioughtcally                                    |26       |26

25/12/24 09:21:18 INFO CodeGenerator: Code generated in 5.066484 ms
25/12/24 09:33:09 INFO BlockManagerInfo: Removed broadcast_16_piece0 on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local:40303 in memory (size: 21.1 KiB, free: 434.3 MiB)
25/12/24 09:33:09 INFO BlockManagerInfo: Removed broadcast_13_piece0 on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local:40303 in memory (size: 33.3 KiB, free: 434.4 MiB)
25/12/24 09:33:09 INFO BlockManagerInfo: Removed broadcast_12_piece0 on dlh-jh-3eb7e5c349c310-jupyterhub-0.mcs.local:40303 in memory (size: 33.3 KiB, free: 434.4 MiB)


In [22]:
# Запись результата в новую таблицу
(
    agg.writeTo("ice.dbt_test.customers_by_country_year")
       .using("iceberg")
       # Здесь можно задать намного больше опций чем в Trino!
        .tableProperty("format-version", "2")
       .createOrReplace()
)

25/12/25 08:07:09 INFO BaseMetastoreCatalog: Table properties set at catalog level through catalog properties: {}
25/12/25 08:07:09 INFO BaseMetastoreCatalog: Table properties enforced at catalog level through catalog properties: {}
25/12/25 08:07:09 INFO V2ScanRelationPushDown: 
Pushing operators to ice.tpcds.item
Pushed Filters: i_category_id IS NOT NULL, i_category_id = 3
Post-Scan Filters: isnotnull(i_category_id#429),(i_category_id#429 = 3)
         
25/12/25 08:07:09 INFO V2ScanRelationPushDown: 
Output: i_item_sk#418L, i_category_id#429, i_manufact#432
         
25/12/25 08:07:09 INFO SnapshotScan: Scanning table ice.tpcds.item snapshot 5313358808039063702 created at 2025-11-24T16:00:43.700+00:00 with filter (i_category_id IS NOT NULL AND i_category_id = (1-digit-int))
25/12/25 08:07:09 INFO BaseDistributedDataScan: Planning file tasks locally for table ice.tpcds.item
25/12/25 08:07:09 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table i